# Deepspeed
ZeRO 将显存优化分为三个递进的阶段，关键是逐步把**优化器状态（Optimizer States）**、**梯度（Gradients）** 和 **模型参数（Parameters）** 从“每卡完整复制”变为“多卡分片存储”。

在经典数据并行下，每张 GPU 需要维护：

$$
\text{Mem}_{\text{DP}} \propto OS + G + P \tag{13.17}
$$

其中，$\text{Mem}_{\text{DP}}$ 表示在经典数据并行（Data Parallel, DP）设定下**单张 GPU 上用于存储模型状态的显存开销**，而 $OS$、$G$、$P$ 分别代表优化器状态、梯度和参数，这里只关注它们的相对规模，所以用“$\propto$”表示与这三部分之和成正比。

这个过程中 ZeRO 分别在三个 Stage 中逐步把这三类状态打散到多张 GPU 上：

（1）**ZeRO-1（Optimizer States Sharding）**

在这一阶段，ZeRO **仅对优化器状态（OS）进行分片**，而参数（P）与梯度（G）仍然在每张 GPU 上完整保存。此时，单卡显存开销近似变为：

$$
\text{Mem}_{\text{ZeRO-1}} \propto P + G + \frac{OS}{DP} \tag{13.18}
$$

相当于大家依然各自持有完整的模型和梯度，但优化器状态（如 Adam 的动量）被切分了，每张卡只负责维护其中一小块。由于优化器状态在总显存中占比往往最大（约 75%），仅这一步就能带来显著的显存节省。这也是为什么在当前的大模型训练实践中，ZeRO-1 往往被视为“最低配置”或默认开启的基线功能。

（2）**ZeRO-2（OS + G 分片）**

在 ZeRO-1 的基础上，ZeRO-2 进一步对**梯度**进行了分片，此时只有参数仍完整复制在每张卡上。单卡显存进一步降低为：

$$
\text{Mem}_{\text{ZeRO-2}} \propto P + \frac{G}{DP} + \frac{OS}{DP} \tag{13.19}
$$

这通常能带来约 8～16 倍的显存节省（视 DP 并行度而定），使得单卡能训练更大的模型。

（3）**ZeRO-3（OS + G + P 全分片）**

这是“火力全开”的终极形态，对**模型参数**本身也进行了分片。此时每张 GPU 只保存整个模型参数的一部分，单卡显存变为：

$$
\text{Mem}_{\text{ZeRO-3}} \propto \frac{P}{DP} + \frac{G}{DP} + \frac{OS}{DP} \tag{13.20}
$$

在分片带来显存红利的同时，也会引入额外的通信开销。ZeRO 论文中对通信量（Communication Volume）进行了详细分析（假设模型参数量为 $\Psi$）：

- **ZeRO-1 / ZeRO-2**：通信量为 **$2\Psi$**，与标准数据并行同阶。直观理解是把标准 DP 的 `all-reduce` 用 `reduce-scatter + all-gather` 这类等价组合来实现后，总通信量量级不变，但通信“形态”发生了变化（例如 ZeRO-2 需要配合分片优化器在 step 内做参数分片的聚合/同步）。
- **ZeRO-3**：通信量为 **$3\Psi$**，约为标准数据并行的 **1.5 倍**。这是因为在前向/反向传播中需要为计算临时聚合当前层所需参数分片（可理解为额外的参数 `all-gather`），再叠加梯度的 `reduce-scatter` 与更新后参数的同步通信。虽然通信量上升，但换来的是**参数级别的全分片**，使得单卡显存占用能随 GPU 数量近似按 $1/DP$ 下降。